# Phase 5 — Statistical Robustness and Error Analysis (Standalone)

This notebook is fully standalone. It recreates the primary Phase 4 predictions and then performs:
- paired bootstrap 95% confidence intervals for Macro-F1 differences;
- McNemar tests;
- seed-consistency analysis;
- false-positive and false-negative summaries.

Use a **T4 GPU** in Colab.


In [ ]:
!pip install -q transformers datasets accelerate pyarrow openpyxl requests scipy scikit-learn


In [ ]:
import re, hashlib, random, warnings, inspect, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch, requests

from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from scipy.stats import chi2, binomtest

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)

warnings.filterwarnings("ignore")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT=Path("/content/stress_detection_phase5")
RAW=ROOT/"raw"
PRED=ROOT/"predictions"
OUT=ROOT/"results"
for p in [RAW,PRED,OUT]:
    p.mkdir(parents=True,exist_ok=True)


## Persistent and restart-safe execution

This version requires a GPU and saves RoBERTa checkpoints and predictions to Google Drive. If Colab disconnects or resets, rerun the notebook: completed seeds are skipped, and an interrupted seed resumes from the latest saved checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Use Runtime → Change runtime type → T4 GPU, then rerun.')

PERSIST=Path('/content/drive/MyDrive/stress_detection_phase5_persistent')
PERSIST_PRED=PERSIST/'predictions'
PERSIST_RESULTS=PERSIST/'results'
PERSIST_CKPT=PERSIST/'checkpoints'
for p in [PERSIST,PERSIST_PRED,PERSIST_RESULTS,PERSIST_CKPT]:
    p.mkdir(parents=True,exist_ok=True)
print('Persistent folder:',PERSIST)
print('GPU:',torch.cuda.get_device_name(0))


## 1. Build leakage-clean Dreaddit


In [ ]:
def norm_text(x):
    if pd.isna(x): return ""
    x=str(x).lower().strip()
    x=re.sub(r"https?://\S+|www\.\S+","<url>",x)
    return re.sub(r"\s+"," ",x)

def h(x):
    return hashlib.sha256(norm_text(x).encode("utf-8")).hexdigest()

def prep(df):
    out=df.copy()
    out["text"]=out["text"].astype("string").str.strip()
    out=out.dropna(subset=["text","label"]).copy()
    out["label"]=pd.to_numeric(out["label"],errors="coerce").astype(int)
    out["text_hash"]=out["text"].map(h)
    return out

ds=load_dataset("andreagasparini/dreaddit")
tr=prep(ds["train"].to_pandas()[["text","label"]])
te=prep(ds["test"].to_pandas()[["text","label"]])

overlap=set(tr["text_hash"]) & set(te["text_hash"])
clean=tr.drop_duplicates("text_hash",keep="first")
clean=clean[~clean["text_hash"].isin(overlap)].reset_index(drop=True)
te=te.reset_index(drop=True)

print("Leakage-clean train:",len(clean))
print("Official test:",len(te))


## 2. Load and deduplicate external datasets


In [ ]:
FILES={
    "Reddit_Title":"Reddit_Title.xlsx",
    "Reddit_Combi":"Reddit_Combi.xlsx",
    "Twitter_Full":"Twitter_Full.xlsx",
    "Twitter_NonAdvert":"Twitter_Non-Advert.xlsx",
}
BASE="https://raw.githubusercontent.com/SenticNet/stress-detection/master"

external={}

for name,fn in FILES.items():
    p=RAW/fn
    if not p.exists():
        r=requests.get(f"{BASE}/{fn}",timeout=60)
        r.raise_for_status()
        p.write_bytes(r.content)

    raw=pd.read_excel(p)
    if name=="Reddit_Title":
        text,label=raw["title"],raw["label"]
    elif name=="Reddit_Combi":
        text=(raw["title"].fillna("").astype(str).str.strip()+" "+
              raw["body"].fillna("").astype(str).str.strip()).str.strip()
        label=raw["label"]
    elif name=="Twitter_Full":
        text,label=raw["text"],raw["labels"]
    else:
        text,label=raw["text"],raw["label"]

    df=prep(pd.DataFrame({"text":text,"label":label}))
    counts=df.groupby("text_hash")["label"].nunique()
    conflicts=set(counts[counts>1].index)
    df=df[~df["text_hash"].isin(conflicts)].drop_duplicates("text_hash",keep="first").reset_index(drop=True)
    external[name]=df

PRIMARY={"Dreaddit_Test":te,**external}

pd.DataFrame([
    {"dataset":k,"rows":len(v),"label_0":int((v.label==0).sum()),"label_1":int((v.label==1).sum())}
    for k,v in PRIMARY.items()
])


## 3. Train Multinomial NB and save predictions


In [ ]:
tfidf=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=0.98,max_features=10000)
X=tfidf.fit_transform(clean["text"].astype(str))
nb=MultinomialNB(alpha=0.25).fit(X,clean["label"])

for name,df in PRIMARY.items():
    Xt=tfidf.transform(df["text"].astype(str))
    pred=nb.predict(Xt)
    df_pred=pd.DataFrame({
        "text":df["text"].reset_index(drop=True),
        "true_label":df["label"].reset_index(drop=True),
        "predicted_label":pred
    })
    df_pred.to_csv(PRED/f"NB_{name}.csv",index=False)
    df_pred.to_csv(PERSIST_PRED/f"NB_{name}.csv",index=False)

print("NB predictions saved.")


## 4. Train RoBERTa across three seeds and save predictions


In [ ]:
MODEL="roberta-base"
SEEDS=[42,52,62]
MAXLEN=256

train_df,val_df=train_test_split(
    clean,test_size=0.10,random_state=42,stratify=clean["label"]
)
train_df=train_df.reset_index(drop=True)
val_df=val_df.reset_index(drop=True)

tok=AutoTokenizer.from_pretrained(MODEL,use_fast=True)
collator=DataCollatorWithPadding(tokenizer=tok)

def tokdf(df):
    hf=Dataset.from_pandas(df[["text","label"]],preserve_index=False)
    return hf.map(lambda b:tok(b["text"],truncation=True,max_length=MAXLEN),batched=True)

train_tok=tokdf(train_df)
val_tok=tokdf(val_df)
test_toks={k:tokdf(v) for k,v in PRIMARY.items()}

def comp_metrics(ep):
    logits,labels=ep
    p=np.argmax(logits,axis=1)
    return {"f1_macro":f1_score(labels,p,average="macro",zero_division=0)}

def args_for(seed):
    kw=dict(
        output_dir=str(PERSIST_CKPT/f"seed_{seed}"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=2,
        seed=seed,
        data_seed=42,
        report_to="none",
        fp16=True,
    )
    sig=inspect.signature(TrainingArguments.__init__)
    kw["eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"]="epoch"
    return TrainingArguments(**kw)

def make_trainer(model,args):
    kw=dict(model=model,args=args,train_dataset=train_tok,eval_dataset=val_tok,
            data_collator=collator,compute_metrics=comp_metrics)
    sig=inspect.signature(Trainer.__init__)
    if "processing_class" in sig.parameters:
        kw["processing_class"]=tok
    elif "tokenizer" in sig.parameters:
        kw["tokenizer"]=tok
    return Trainer(**kw)

for seed in SEEDS:
    complete=PERSIST_PRED/f"RB_{seed}_Dreaddit_Test.csv"
    if complete.exists():
        print(f"Seed {seed} already completed. Skipping training.")
        for name in PRIMARY:
            pf=PERSIST_PRED/f"RB_{seed}_{name}.csv"
            if pf.exists():
                pd.read_csv(pf).to_csv(PRED/f"RB_{seed}_{name}.csv",index=False)
        continue

    print("Training seed",seed)
    set_seed(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=2)
    trainer=make_trainer(model,args_for(seed))

    ckpt_dir=PERSIST_CKPT/f"seed_{seed}"
    checkpoints=sorted(
        ckpt_dir.glob("checkpoint-*"),
        key=lambda p:int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1
    )
    resume=str(checkpoints[-1]) if checkpoints else None
    if resume:
        print("Resuming from",resume)
        trainer.train(resume_from_checkpoint=resume)
    else:
        trainer.train()

    for name,dset in test_toks.items():
        out=trainer.predict(dset)
        p=np.argmax(out.predictions,axis=1)
        df_pred=pd.DataFrame({
            "text":PRIMARY[name]["text"].reset_index(drop=True),
            "true_label":out.label_ids,
            "predicted_label":p
        })
        df_pred.to_csv(PRED/f"RB_{seed}_{name}.csv",index=False)
        df_pred.to_csv(PERSIST_PRED/f"RB_{seed}_{name}.csv",index=False)

    del trainer,model
    gc.collect()
    torch.cuda.empty_cache()

print("RoBERTa predictions saved.")


## 5. Paired bootstrap and McNemar tests


In [ ]:
# Restore persistent predictions into local analysis folder if needed.
for name in PRIMARY:
    nbp=PERSIST_PRED/f"NB_{name}.csv"
    if nbp.exists():
        pd.read_csv(nbp).to_csv(PRED/f"NB_{name}.csv",index=False)
    for seed in SEEDS:
        rbp=PERSIST_PRED/f"RB_{seed}_{name}.csv"
        if rbp.exists():
            pd.read_csv(rbp).to_csv(PRED/f"RB_{seed}_{name}.csv",index=False)

NBOOT=2000
RSEED=2026

def mf1(y,p):
    return f1_score(y,p,average="macro",zero_division=0)

def boot_diff(y,rb,nb,seed):
    y=np.asarray(y); rb=np.asarray(rb); nb=np.asarray(nb)
    rng=np.random.default_rng(seed)
    n=len(y)
    vals=np.empty(NBOOT)
    for i in range(NBOOT):
        idx=rng.integers(0,n,size=n)
        vals[i]=mf1(y[idx],rb[idx])-mf1(y[idx],nb[idx])
    point=mf1(y,rb)-mf1(y,nb)
    p=min(1.0,2*min(np.mean(vals<=0),np.mean(vals>=0)))
    return point,np.percentile(vals,2.5),np.percentile(vals,97.5),p

def mcnemar(y,rb,nb):
    y=np.asarray(y); rb=np.asarray(rb); nb=np.asarray(nb)
    rbc=rb==y; nbc=nb==y
    b=int(np.sum((~rbc)&nbc))
    c=int(np.sum(rbc&(~nbc)))
    d=b+c
    if d==0: return b,c,d,np.nan,1.0
    if d<25:
        return b,c,d,np.nan,binomtest(min(b,c),n=d,p=0.5,alternative="two-sided").pvalue
    stat=((abs(b-c)-1)**2)/d
    return b,c,d,stat,chi2.sf(stat,1)

boot_rows=[]
mc_rows=[]

for name in PRIMARY:
    nbdf=pd.read_csv(PRED/f"NB_{name}.csv")
    y=nbdf["true_label"].to_numpy()
    pnb=nbdf["predicted_label"].to_numpy()

    for seed in SEEDS:
        rbdf=pd.read_csv(PRED/f"RB_{seed}_{name}.csv")
        prb=rbdf["predicted_label"].to_numpy()

        d,lo,hi,p=boot_diff(y,prb,pnb,RSEED+seed)
        boot_rows.append({
            "dataset":name,"seed":seed,
            "f1_diff_roberta_minus_nb":d,
            "ci_low":lo,"ci_high":hi,
            "bootstrap_p_two_sided":p
        })

        b,c,disc,stat,pv=mcnemar(y,prb,pnb)
        mc_rows.append({
            "dataset":name,"seed":seed,
            "nb_correct_rb_wrong":b,
            "nb_wrong_rb_correct":c,
            "discordant":disc,
            "statistic":stat,
            "p_value":pv
        })

boot=pd.DataFrame(boot_rows)
mc=pd.DataFrame(mc_rows)

print("Paired bootstrap:")
display(boot.round(5))
print("McNemar:")
display(mc.round(6))

boot.to_csv(OUT/"paired_bootstrap.csv",index=False)
mc.to_csv(OUT/"mcnemar.csv",index=False)

boot.to_csv(PERSIST_RESULTS/"paired_bootstrap.csv",index=False)
mc.to_csv(PERSIST_RESULTS/"mcnemar.csv",index=False)


## 6. Seed consistency and error summary


In [ ]:
seed_summary=(boot.groupby("dataset")
    .agg(
        mean_f1_difference=("f1_diff_roberta_minus_nb","mean"),
        min_f1_difference=("f1_diff_roberta_minus_nb","min"),
        max_f1_difference=("f1_diff_roberta_minus_nb","max"),
        seeds_favoring_roberta=("f1_diff_roberta_minus_nb",lambda s:int((s>0).sum())),
        seeds_favoring_nb=("f1_diff_roberta_minus_nb",lambda s:int((s<0).sum())),
    ).reset_index())

display(seed_summary.round(4))

error_rows=[]
for name in PRIMARY:
    nbdf=pd.read_csv(PRED/f"NB_{name}.csv")
    for model,seed,df in [("Multinomial NB",np.nan,nbdf)]+[
        ("RoBERTa-base",s,pd.read_csv(PRED/f"RB_{s}_{name}.csv")) for s in SEEDS
    ]:
        fp=int(((df.true_label==0)&(df.predicted_label==1)).sum())
        fn=int(((df.true_label==1)&(df.predicted_label==0)).sum())
        err=(fp+fn)/len(df)
        error_rows.append({"model":model,"seed":seed,"dataset":name,
                           "false_positives":fp,"false_negatives":fn,"error_rate":err})

errors=pd.DataFrame(error_rows)
display(errors.round(4))

seed_summary.to_csv(OUT/"seed_consistency.csv",index=False)
errors.to_csv(OUT/"error_summary.csv",index=False)

seed_summary.to_csv(PERSIST_RESULTS/"seed_consistency.csv",index=False)
errors.to_csv(PERSIST_RESULTS/"error_summary.csv",index=False)


## 7. Primary statistical summary


In [ ]:
rows=[]
for name in PRIMARY:
    b=boot[boot.dataset==name]
    m=mc[mc.dataset==name]
    rows.append({
        "dataset":name,
        "mean_macro_f1_difference_RB_minus_NB":b.f1_diff_roberta_minus_nb.mean(),
        "all_seeds_same_direction":bool(
            (b.f1_diff_roberta_minus_nb>0).all() or (b.f1_diff_roberta_minus_nb<0).all()
        ),
        "all_bootstrap_CIs_exclude_zero":bool(
            ((b.ci_low>0)&(b.ci_high>0)).all() or ((b.ci_low<0)&(b.ci_high<0)).all()
        ),
        "all_mcnemar_p_lt_0.05":bool((m.p_value<0.05).all()),
        "median_mcnemar_p":m.p_value.median(),
    })
summary=pd.DataFrame(rows)
display(summary.round(6))
summary.to_csv(OUT/"primary_statistical_summary.csv",index=False)

summary.to_csv(PERSIST_RESULTS/"primary_statistical_summary.csv",index=False)


## Send me screenshots of:
1. Paired bootstrap table
2. McNemar table
3. Seed-consistency table
4. Error-summary table
5. Primary statistical summary
